# Event-TimeRAF: Los Angeles County PM2.5 Pipeline

This notebook executes the official-source, leakage-safe experiment defined in `structured_plan.md`. It downloads no synthetic research data and stops when a configured readiness gate fails.

In [ ]:
# Kaggle setup: uncomment only when the packages are not already available.
# %pip install -q pyarrow holidays xgboost shap chronos-forecasting


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'default.yaml').exists():
    if (PROJECT_ROOT.parent / 'configs' / 'default.yaml').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError('Set PROJECT_ROOT to the repository working copy.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from event_timeraf.config import load_config
from event_timeraf.data import (
    build_data_audit, download_epa_pm25,
    download_storm_events, load_optional_hms_events, prepare_epa_pm25,
    prepare_storm_events, select_and_download_noaa_weather,
    write_run_manifest,
)
from event_timeraf.features import build_modeling_table
from event_timeraf.windows import build_window_dataset
from event_timeraf.retrieval import HistoricalRetriever, build_knowledge_base
from event_timeraf.models import (
    DirectXGBForecaster, choose_fusion_weight, chronos_forecast,
    daily_seasonal_forecast, fuse_forecasts, origin_feature_matrix,
    persistence_forecast, weekly_seasonal_forecast,
)
from event_timeraf.drift import DriftDetector
from event_timeraf.evaluation import (
    metric_values, metrics_table, paired_block_bootstrap_difference, predictions_long,
)
from event_timeraf.explain import generate_explanations, xgb_local_contributions
from event_timeraf.plots import plot_forecast_case, plot_horizon_metrics

cfg = load_config(PROJECT_ROOT / 'configs' / 'default.yaml', PROJECT_ROOT)
FORCE_DOWNLOAD = False
REQUIRE_EVENTS = True
REQUIRE_STRICT_EVENT_AVAILABILITY = False
RUN_TSF_MODEL = False
HMS_CACHE = None  # Optional source-preserving CSV/Parquet path.
np.random.seed(cfg.seed)
cfg


## 1. Official data acquisition and readiness audit

EPA national ZIP files are cached and filtered by state/county while reading in chunks. NOAA weather and event files are also cached so the prepared `data/raw` directory can be attached to a later Kaggle run without internet.

In [ ]:
epa_raw = download_epa_pm25(cfg, force=FORCE_DOWNLOAD)
pm25, site_coverage = prepare_epa_pm25(epa_raw, cfg)
best_site = site_coverage.iloc[0]

weather_station, weather_raw, weather = select_and_download_noaa_weather(
    cfg, float(best_site['latitude']), float(best_site['longitude']), force=FORCE_DOWNLOAD
)

storm_raw = download_storm_events(cfg, force=FORCE_DOWNLOAD)
events = prepare_storm_events(storm_raw, cfg)
if HMS_CACHE is not None:
    hms_events = load_optional_hms_events(HMS_CACHE, cfg)
    events = pd.concat([events, hms_events], ignore_index=True).drop_duplicates('event_id')

pm25.to_parquet(cfg.paths.processed / 'la_pm25_hourly.parquet', index=False)
weather.to_parquet(cfg.paths.processed / 'la_weather_hourly.parquet', index=False)
events.to_parquet(cfg.paths.knowledge_base / 'event_kb.parquet', index=False)
audit = build_data_audit(pm25, weather, events, cfg, site_coverage, weather_station)
display(pd.DataFrame([audit['gates']]))
if not audit['core_ready']:
    raise RuntimeError('Core data-readiness gate failed. Inspect outputs/audit/data_audit.json.')
if REQUIRE_EVENTS and not audit['event_ready']:
    raise RuntimeError('Event-data gate failed. Add an audited HMS cache or revise event claims.')
if REQUIRE_STRICT_EVENT_AVAILABILITY and not audit['strict_event_availability']:
    raise RuntimeError('Strict event availability failed: genuine publication timestamps are required.')
if REQUIRE_EVENTS and not audit['strict_event_availability']:
    warnings.warn('Event-aware outputs are retrospective availability sensitivity results.')
audit


## 2. Causal features and 168-to-24 windows

In [ ]:
modeling = build_modeling_table(pm25, weather, events, cfg)
modeling_path = cfg.paths.processed / 'modeling_hourly.parquet'
modeling.to_parquet(modeling_path, index=False)

dataset = build_window_dataset(modeling, cfg)
dataset.save(
    cfg.paths.processed / 'window_arrays.npz',
    cfg.paths.processed / 'window_metadata.parquet',
)
train = dataset.subset('train')
validation = dataset.subset('validation')
test = dataset.subset('test')
print({'train': len(train.x), 'validation': len(validation.x), 'test': len(test.x)})
print('X/Y shapes:', dataset.x.shape, dataset.y.shape)


## 3. Leakage-safe historical retrieval

In [ ]:
knowledge_base = build_knowledge_base(dataset, cfg)
knowledge_base.save(
    cfg.paths.knowledge_base / 'ts_kb_arrays.npz',
    cfg.paths.knowledge_base / 'ts_kb_metadata.parquet',
)
retriever = HistoricalRetriever(knowledge_base, cfg)

cosine_train = retriever.retrieve(train, method='cosine')
cosine_validation = retriever.retrieve(validation, method='cosine')
cosine_test = retriever.retrieve(test, method='cosine')
hybrid_train = retriever.retrieve(train, method='hybrid')
hybrid_validation = retriever.retrieve(validation, method='hybrid')
hybrid_test = retriever.retrieve(test, method='hybrid')
random_test = retriever.retrieve(test, method='random')

if not (cosine_test.valid_mask.all() and hybrid_test.valid_mask.all() and random_test.valid_mask.all()):
    raise RuntimeError('A test query has no causally eligible retrieval candidate.')
evidence = pd.concat(
    [cosine_test.evidence, hybrid_test.evidence, random_test.evidence], ignore_index=True
)
evidence.to_parquet(cfg.paths.outputs / 'evidence' / 'retrieval_evidence.parquet', index=False)
print('Knowledge-base candidates:', len(knowledge_base.metadata))
display(hybrid_test.evidence.head())


## 4. Baselines and Event-TimeRAF variants

Each XGBoost model uses 24 direct regressors. Retrieval-augmented training excludes only early training origins that have no causally eligible knowledge-base candidate; all reported test models use the same test windows.

In [ ]:
predictions = {
    'M00_persistence': persistence_forecast(test.x, cfg.forecast.horizon),
    'M01_daily_seasonal': daily_seasonal_forecast(test.x, cfg.forecast.horizon),
    'M02_weekly_seasonal': weekly_seasonal_forecast(test.x, cfg.forecast.horizon),
    'M05_random_retrieval': random_test.prediction,
    'M06_cosine_retrieval': cosine_test.prediction,
}

pm_train, pm_names = origin_feature_matrix(train, ('pm25_',))
pm_test, _ = origin_feature_matrix(test, ('pm25_',))
m03 = DirectXGBForecaster(cfg, include_future_calendar=False).fit(
    pm_train, train.future_calendar, train.y, pm_names, train.calendar_names
)
predictions['M03_xgb_pm25'] = m03.predict(pm_test, test.future_calendar)

context_prefixes = ('pm25_', 'weather_', 'cal_')
context_train, context_names = origin_feature_matrix(train, context_prefixes)
context_test, _ = origin_feature_matrix(test, context_prefixes)
m04 = DirectXGBForecaster(cfg, include_future_calendar=True).fit(
    context_train, train.future_calendar, train.y, context_names, train.calendar_names
)
predictions['M04_xgb_context'] = m04.predict(context_test, test.future_calendar)

cosine_train_mask = cosine_train.valid_mask
retrieval_prefixes = ('pm25_', 'weather_', 'cal_')
m07_train, m07_names = origin_feature_matrix(train, retrieval_prefixes, cosine_train.as_features())
m07_test, _ = origin_feature_matrix(test, retrieval_prefixes, cosine_test.as_features())
m07 = DirectXGBForecaster(cfg, include_future_calendar=True).fit(
    m07_train[cosine_train_mask], train.future_calendar[cosine_train_mask], train.y[cosine_train_mask],
    m07_names, train.calendar_names,
)
predictions['M07_xgb_cosine'] = m07.predict(m07_test, test.future_calendar)

full_prefixes = ('pm25_', 'weather_', 'cal_', 'event_')
hybrid_train_mask = hybrid_train.valid_mask
m08_train, m08_names = origin_feature_matrix(train, full_prefixes, hybrid_train.as_features())
m08_test, _ = origin_feature_matrix(test, full_prefixes, hybrid_test.as_features())
m08 = DirectXGBForecaster(cfg, include_future_calendar=True).fit(
    m08_train[hybrid_train_mask], train.future_calendar[hybrid_train_mask], train.y[hybrid_train_mask],
    m08_names, train.calendar_names,
)
predictions['M08_event_timeraf_no_drift'] = m08.predict(m08_test, test.future_calendar)


In [ ]:
drift_detector = DriftDetector(cfg).fit_reference(train, hybrid_train.mean_similarity)
drift_detector.calibrate(validation, hybrid_validation.mean_similarity)
drift_train = drift_detector.transform(train, hybrid_train.mean_similarity)
drift_validation = drift_detector.transform(validation, hybrid_validation.mean_similarity)
drift_test = drift_detector.transform(test, hybrid_test.mean_similarity)

train_extra = np.column_stack([hybrid_train.as_features(), drift_train.components, drift_train.score])
test_extra = np.column_stack([hybrid_test.as_features(), drift_test.components, drift_test.score])
m09_train, m09_names = origin_feature_matrix(train, full_prefixes, train_extra)
m09_test, _ = origin_feature_matrix(test, full_prefixes, test_extra)
m09 = DirectXGBForecaster(cfg, include_future_calendar=True).fit(
    m09_train[hybrid_train_mask], train.future_calendar[hybrid_train_mask], train.y[hybrid_train_mask],
    m09_names, train.calendar_names,
)
predictions['M09_event_timeraf_full'] = m09.predict(m09_test, test.future_calendar)
for name, model in {'M03': m03, 'M04': m04, 'M07': m07, 'M08': m08, 'M09': m09}.items():
    model.save(cfg.paths.outputs / 'models' / f'{name}.joblib')
print('Drift threshold:', drift_test.threshold, 'flagged test origins:', int(drift_test.flag.sum()))


## 5. Evaluation, saved evidence, and grounded explanations

In [ ]:
event_index = test.feature_names.index('event_count_72h')
event_flag = test.features[:, event_index] > 0
subset_masks = {
    'event': event_flag,
    'non_event': ~event_flag,
    'drift': drift_test.flag,
    'non_drift': ~drift_test.flag,
}
def model_metric_frames(name, values):
    frames = [metrics_table(test.y, values, name)]
    for subset_name, mask in subset_masks.items():
        if mask.any():
            frames.append(metrics_table(test.y[mask], values[mask], name, subset=subset_name))
    return frames

metric_frames = [
    frame
    for name, values in predictions.items()
    for frame in model_metric_frames(name, values)
]
metrics = pd.concat(metric_frames, ignore_index=True)
prediction_frames = [
    predictions_long(test.y, values, test.metadata, name, drift_test.flag, event_flag)
    for name, values in predictions.items()
]
prediction_table = pd.concat(prediction_frames, ignore_index=True)
metrics.to_csv(cfg.paths.outputs / 'tables' / 'metrics.csv', index=False)
metrics.loc[(metrics['horizon'] == 'overall') & (metrics['subset'] == 'all')].to_csv(
    cfg.paths.outputs / 'tables' / 'main_results.csv', index=False
)
prediction_table.to_parquet(cfg.paths.outputs / 'predictions' / 'predictions.parquet', index=False)
display(metrics.loc[(metrics['horizon'] == 'overall') & (metrics['subset'] == 'all')].sort_values('mse'))

k_rows = []
for candidate_k in cfg.retrieval.k_values:
    result_k = retriever.retrieve(test, method='cosine', k=candidate_k)
    k_rows.append({'k': candidate_k, **metric_values(test.y, result_k.prediction)})
pd.DataFrame(k_rows).to_csv(
    cfg.paths.outputs / 'tables' / 'k_sensitivity_results.csv', index=False
)
fusion_rows = []
for method, result in {'cosine': cosine_test, 'hybrid': hybrid_test}.items():
    fusion_rows.append({'method': method, 'aggregation': 'uniform', **metric_values(test.y, result.prediction)})
    fusion_rows.append({'method': method, 'aggregation': 'similarity_weighted', **metric_values(test.y, result.weighted_prediction)})
pd.DataFrame(fusion_rows).to_csv(
    cfg.paths.outputs / 'tables' / 'retrieval_fusion_ablation.csv', index=False
)
bootstrap_rows = []
for metric_name, metric_fn in {
    'mse': lambda y, p: float(np.mean((y - p) ** 2)),
    'mae': lambda y, p: float(np.mean(np.abs(y - p))),
}.items():
    comparison = paired_block_bootstrap_difference(
        test.y, predictions['M09_event_timeraf_full'], predictions['M04_xgb_context'],
        metric_fn, cfg.evaluation.bootstrap_block_hours,
        cfg.evaluation.bootstrap_resamples, cfg.seed,
    )
    bootstrap_rows.append({'comparison': 'M09_minus_M04', 'metric': metric_name, **comparison})
pd.DataFrame(bootstrap_rows).to_csv(
    cfg.paths.outputs / 'tables' / 'ablation_results.csv', index=False
)

h1_matrix = m09._matrix(m09_test, test.future_calendar, 0)
contributions = xgb_local_contributions(m09.models[0], h1_matrix)
explanations = generate_explanations(
    test, predictions['M09_event_timeraf_full'], hybrid_test, drift_test, events,
    contributions, m09.feature_names,
)
explanations.to_parquet(cfg.paths.outputs / 'evidence' / 'explanations.parquet', index=False)
display(explanations.head(3))


## 6. Optional frozen-TSFM publication gate

This cell is deliberately opt-in because it downloads model weights. It evaluates the same 168-hour context and 24-hour target as every other model. The fusion weight is selected on validation data only.

In [ ]:
if RUN_TSF_MODEL:
    tsfm_validation, tsfm_val_low, tsfm_val_high = chronos_forecast(
        validation.x, cfg.forecast.horizon, cfg.tsfm.checkpoint, cfg.tsfm.batch_size
    )
    tsfm_test, tsfm_test_low, tsfm_test_high = chronos_forecast(
        test.x, cfg.forecast.horizon, cfg.tsfm.checkpoint, cfg.tsfm.batch_size
    )
    selected_weight, fusion_scores = choose_fusion_weight(
        validation.y, tsfm_validation, hybrid_validation.prediction, cfg.tsfm.fusion_weights
    )
    predictions['M10_frozen_chronos'] = tsfm_test
    predictions['M11_chronos_hybrid_retrieval'] = fuse_forecasts(
        tsfm_test, hybrid_test.prediction, selected_weight
    )
    np.savez_compressed(
        cfg.paths.outputs / 'predictions' / 'tsfm_predictions.npz',
        mean=tsfm_test, lower=tsfm_test_low, upper=tsfm_test_high, fusion_weight=selected_weight,
    )
    tsfm_names = ('M10_frozen_chronos', 'M11_chronos_hybrid_retrieval')
    metrics = pd.concat(
        [metrics] + [frame for name in tsfm_names for frame in model_metric_frames(name, predictions[name])],
        ignore_index=True,
    )
    metrics.to_csv(cfg.paths.outputs / 'tables' / 'metrics.csv', index=False)
    metrics.loc[(metrics['horizon'] == 'overall') & (metrics['subset'] == 'all')].to_csv(
        cfg.paths.outputs / 'tables' / 'main_results.csv', index=False
    )
    prediction_table = pd.concat(
        [prediction_table] + [
            predictions_long(test.y, predictions[name], test.metadata, name, drift_test.flag, event_flag)
            for name in tsfm_names
        ],
        ignore_index=True,
    )
    prediction_table.to_parquet(
        cfg.paths.outputs / 'predictions' / 'predictions.parquet', index=False
    )
    print('Selected TSFM weight:', selected_weight, fusion_scores)
else:
    print('TSFM gate skipped. The final paper must not claim it was completed.')


## 7. Figures and manifest

In [ ]:
plot_horizon_metrics(
    metrics, 'mae', cfg.paths.outputs / 'figures' / 'mae_by_horizon.png'
)
case_index = int(np.argsort(np.abs(test.y.mean(axis=1) - predictions['M09_event_timeraf_full'].mean(axis=1)))[len(test.y) // 2])
plot_forecast_case(
    test.x[case_index], test.y[case_index],
    {
        'Persistence': predictions['M00_persistence'][case_index],
        'XGBoost context': predictions['M04_xgb_context'][case_index],
        'Event-TimeRAF': predictions['M09_event_timeraf_full'][case_index],
    },
    cfg.paths.outputs / 'figures' / 'forecast_case.png',
)
manifest = write_run_manifest(
    cfg,
    [
        modeling_path,
        cfg.paths.processed / 'window_arrays.npz',
        cfg.paths.outputs / 'tables' / 'metrics.csv',
        cfg.paths.outputs / 'predictions' / 'predictions.parquet',
        cfg.paths.outputs / 'evidence' / 'retrieval_evidence.parquet',
    ],
)
print(json.dumps(manifest, indent=2))
